# 03 - Model Training
Train the tracking error model with temporal validation.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for package in ["yfinance", "shap", "plotly"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from config import PAIR_CONFIGS
from src.data_loader import MarketDataLoader
from src.features import FeatureEngineer
from src.models import TrackingErrorModel
from src.utils import time_split

loader   = MarketDataLoader()
panel    = loader.fetch_universe(PAIR_CONFIGS, period="2y", interval="1d")
features = FeatureEngineer(rolling_window=20, horizon=1).transform_universe(panel)

model  = TrackingErrorModel(random_state=42)
result = model.train(features, target_col="target_te", test_size=0.2)
print(f"Train rows: {result.train_rows:,}   |   Test rows: {result.test_rows:,}")
result

c:\Users\Alexandre\anaconda3\envs\catan\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TrainResult(metrics={'mae': 0.0002630855561563704, 'rmse': 0.0007956269309198496, 'r2': 0.9435327012365649, 'mape': 0.08296606201414168}, train_rows=1396, test_rows=350)

In [ ]:
# --- Persist model artifact for downstream notebooks and the dashboard ---------
artifacts_dir = project_root / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / "te_model.joblib"
model.save(model_path)
print(f"Model saved → {model_path}")

'C:\\Users\\Alexandre\\Documents\\code\\ETF-error-tracking\\artifacts\\te_model.joblib'

In [ ]:
# --- Formatted validation metrics ----------------------------------------------
metrics = result.metrics
metrics_df = pd.DataFrame(
    {
        "Metric"      : ["MAE", "RMSE", "MAPE", "R²"],
        "Value"       : [metrics["mae"], metrics["rmse"], metrics["mape"], metrics["r2"]],
        "Description" : [
            "Mean absolute error (raw TE units)",
            "Root mean squared error — penalises tail misses",
            "Mean absolute percentage error",
            "Proportion of variance explained by the model",
        ],
    }
).set_index("Metric")

# Pretty display — R² > 0.85 is considered strong for TE forecasting.
metrics_df.style.format({"Value": "{:.6f}"})

In [ ]:
# --- Out-of-sample: Actual vs Predicted tracking error (scatter) ---------------
target_col = "target_te"
model_df   = features.dropna(subset=[target_col]).copy()
feat_cols  = model.numeric_columns + model.categorical_columns
model_df   = model_df.dropna(subset=feat_cols)

_, test_df = time_split(model_df, test_size=0.2)
x_test     = test_df[feat_cols]
y_test     = test_df[target_col].to_numpy()
y_pred     = model.predict(x_test)

oos_df = pd.DataFrame({"actual": y_test * 100, "predicted": y_pred * 100,
                        "pair": test_df["pair"].values})
axis_max = float(max(oos_df[["actual", "predicted"]].abs().max().max()) * 1.05)

fig = px.scatter(
    oos_df,
    x="actual",
    y="predicted",
    color="pair",
    opacity=0.55,
    title="Out-of-Sample: Actual vs Predicted Tracking Error (%)",
    labels={"actual": "Actual TE (%)", "predicted": "Predicted TE (%)"},
    template="plotly_white",
    height=480,
)
# Perfect-prediction reference line
fig.add_trace(go.Scatter(
    x=[-axis_max, axis_max],
    y=[-axis_max, axis_max],
    mode="lines",
    line=dict(color="black", dash="dash", width=1),
    name="Perfect prediction",
    showlegend=True,
))
fig.update_xaxes(range=[-axis_max, axis_max])
fig.update_yaxes(range=[-axis_max, axis_max])
fig.show()

In [ ]:
# --- SHAP feature importance bar (top 15 contributors) ------------------------
# Uses the model's built-in explain_shap which computes mean |SHAP| per feature.
# A sample of recent rows is used to keep computation fast in the notebook.
recent_x   = test_df[feat_cols].tail(200)
shap_df    = model.explain_shap(recent_x, max_samples=150).head(15)

fig2 = px.bar(
    shap_df,
    x="mean_abs_shap",
    y="feature",
    orientation="h",
    title="SHAP Feature Importance – Top 15 (mean |SHAP value| on OOS sample)",
    labels={"mean_abs_shap": "Mean |SHAP|", "feature": "Feature"},
    color="mean_abs_shap",
    color_continuous_scale=["#e8f0fc", "#103a6f"],
    template="plotly_white",
    height=500,
)
fig2.update_layout(coloraxis_showscale=False, yaxis=dict(autorange="reversed"))
fig2.show()